Modeling: Mass Total + Source Parametric
========================================

This script fits an `Interferometer` and `Imaging` dataset of a 'galaxy-scale' strong lens with a model where:

 - The lens galaxy's light is an MGE (but is invisible in the interferometer data).
 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField`.
 - The source galaxy's light is an MGE.

__Contents__

- **Benefits:** A number of benefits are apparently if we combine the analysis of both datasets at both wavelengths.
- **Interferometer Masking:** We define the ‘real_space_mask’ which defines the grid the image the strong lens is evaluated using.
- **Interferometer Dataset:** Load and plot the strong lens `Interferometer` dataset `simple__no_lens_light` from .fits files.
- **Imaging Dataset:** Load and plot the strong lens dataset `simple__no_lens_light` via .fits files, which we will fit.
- **Imaging Masking:** Define a 3.0" circular mask, which includes the emission of the lens and source galaxies.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **Model:** Compose the lens model fitted to the data.
- **Shared Source Mesh (Pixelization):** Reconstruct the source of both datasets on one shared Delaunay mesh via `shared_preloads=True`.
- **Search:** Configure the non-linear search used to fit the model.
- **Result:** Overview of the results of the model-fit.

__Benefits__

 A number of benefits are apparently if we combine the analysis of both datasets at both wavelengths:

 - The lens galaxy is invisible at sub-mm wavelengths, making it straight-forward to infer a lens mass model by
 fitting the source at submm wavelengths.

 - The source galaxy appears completely different in the g-band and at sub-millimeter wavelengths, providing a lot
 more information with which to constrain the lens galaxy mass model.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt
import numpy as np

__Interferometer Masking__

We define the ‘real_space_mask’ which defines the grid the image the strong lens is evaluated using.

In [ ]:
mask_radius = 4.0

real_space_mask = al.Mask2D.circular(
    shape_native=(800, 800), pixel_scales=0.05, radius=mask_radius
)

__Interferometer Dataset__

Load and plot the strong lens `Interferometer` dataset `simple__no_lens_light` from .fits files, which we will fit 
with the lens model.

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "interferometer"
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "scripts/multi_dataset/features/imaging_and_interferometer/simulator.py",
        ],
        check=True,
    )

interferometer = al.Interferometer.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    uv_wavelengths_path=dataset_path / "uv_wavelengths.fits",
    real_space_mask=real_space_mask,
    transformer_class=al.TransformerDFT,
)

aplt.subplot_interferometer_dirty_images(dataset=interferometer)

__Imaging Dataset__

Load and plot the strong lens dataset `simple__no_lens_light` via .fits files, which we will fit with the lens model.

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "imaging"
dataset_name = "lens_sersic"
dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/multi_dataset/simulator.py"],
        check=True,
    )

imaging = al.Imaging.from_fits(
    data_path=Path(dataset_path, "g_data.fits"),
    psf_path=Path(dataset_path, "g_psf.fits"),
    noise_map_path=Path(dataset_path, "g_noise_map.fits"),
    pixel_scales=0.08,
)

aplt.subplot_imaging_dataset(dataset=imaging)

__Imaging Masking__

Define a 3.0" circular mask, which includes the emission of the lens and source galaxies.

In [ ]:
mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=imaging.shape_native,
    pixel_scales=imaging.pixel_scales,
    radius=mask_radius,
)

imaging = imaging.apply_mask(mask=mask)

aplt.subplot_imaging_dataset(dataset=imaging)

__Analysis__

We create analysis objects for both datasets.

In [ ]:
analysis_imaging = al.AnalysisImaging(dataset=imaging)
analysis_interferometer = al.AnalysisInterferometer(dataset=interferometer)

__Model__

We compose our lens model using `Model` objects, which represent the galaxies we fit to our data. In this 
example our lens model is:

 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField` [7 parameters].
 -  with 1 x 20 Gaussians for the source galaxy's light, which is complete different for each waveband. [8 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=21.

In [ ]:
bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=True,
    sigma_min=imaging.pixel_scales[0] / 10.0,
)

lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=bulge,
    mass=al.mp.Isothermal,
)

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source),
    fields=field,
)

We now combine them using the factor analysis class, which allows us to fit the two datasets simultaneously.

Imaging and interferometer datasets observe completely different properties of the, such that the galaxy appears 
completely different in the imaging data (e.g. optical emission) and sub-millimeter wavelengths, meaning a completely 
different source model should be used for each dataset.

For this reason, we move lens light and source model composition to the `AnalysisFactor` class, which allows us to fit 
the two datasets simultaneously but with different models.

The benefit of fitting them simultaneously is that the mass model is inferred from both datasets.

In [ ]:
analysis_factor_list = []

for analysis in [analysis_imaging, analysis_interferometer]:

    model_analysis = model.copy()

    model_analysis.galaxies.lens.bulge = af.Model(al.lp_linear.Sersic)

    bulge = al.model_util.mge_model_from(
        mask_radius=mask_radius,
        total_gaussians=20,
        gaussian_per_basis=1,
        centre_prior_is_uniform=False,
    )

    model_analysis.galaxies.source.bulge = bulge

    analysis_factor = af.AnalysisFactor(prior_model=model_analysis, analysis=analysis)

    analysis_factor_list.append(analysis_factor)

factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

The `info` of the model shows us there are two models, one for the imaging dataset and one for the interferometer
dataset. 

In [ ]:
print(factor_graph.global_prior_model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free, 
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the 
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed 
parameter.

In [ ]:
af.ModelPlotter(factor_graph.global_prior_model).figure()

__Shared Source Mesh (Pixelization)__

When the source is reconstructed on a pixelization, the imaging and interferometer fits can share one
source-plane Delaunay mesh by setting `shared_preloads=True` on both analyses (see
`features/same_wavelength/modeling.py` for the full description):

    analysis_list = [
        al.AnalysisImaging(dataset=dataset_imaging, adapt_images=adapt_images, shared_preloads=True),
        al.AnalysisInterferometer(dataset=dataset_interferometer, shared_preloads=True),
    ]

The shared mesh is source-plane geometry, so it is dataset-type-agnostic: the lead factor (the first analysis
in the graph that opted in, of either type) ray-traces it once per likelihood evaluation and both datasets map
their own grids onto it. Every fit consumes only the parts of the shared state that are valid for its dataset
type — the mesh geometry crosses dataset types, while a curvature matrix or mapper shared between identical
interferometer channels (the datacube case) never leaks into an imaging fit. Each dataset keeps its own
reconstruction and its own linear algebra (the imaging fit blurs with its PSF; the interferometer fit works in
the uv-plane), but the two source reconstructions live on the identical source-pixel grid and are directly
comparable.

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("multi_dataset", "modeling"),
    name="imaging_and_interferometer",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result_list = search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)

__Result__

The search returns a result object, which includes: 

 - The lens model corresponding to the maximum log likelihood solution in parameter space.
 - The corresponding maximum log likelihood `Tracer` and `FitInterferometer` objects.
  - Information on the posterior as estimated by the `Nautilus` non-linear search.

In [ ]:
print(result_list[0].max_log_likelihood_instance)

aplt.subplot_tracer(
    tracer=result_list[0].max_log_likelihood_tracer,
    grid=real_space_mask.derive_grid.unmasked,
)

aplt.subplot_fit_imaging(fit=result_list[0].max_log_likelihood_fit)

aplt.subplot_fit_interferometer(fit=result_list[1].max_log_likelihood_fit)
aplt.subplot_fit_interferometer_dirty_images(fit=result_list[1].max_log_likelihood_fit)

aplt.corner_anesthetic(samples=result_list.samples)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.